In [ ]:
# ==========================================================
# CELL 1 — INSTALL REQUIRED PACKAGES
# ==========================================================

!pip install -q \
    groq \
    chromadb \
    langchain-chroma \
    langchain-huggingface \
    langchain-community \
    sentence-transformers \
    rank-bm25 \
    openpyxl


In [ ]:
# ==========================================================
# CELL 2 — SAFE GOOGLE DRIVE CONNECTION
# ==========================================================

import os
from pathlib import Path

# ----------------------------------------------------------
# First try existing Drive mounts.
# DO NOT delete or clear any mountpoint.
# ----------------------------------------------------------

possible_drive_roots = [
    "/content/drive/MyDrive",
    "/content/gdrive/MyDrive",
    "/content/gdrive_safe/MyDrive"
]

DRIVE_ROOT = None

for candidate in possible_drive_roots:
    if os.path.isdir(candidate):
        DRIVE_ROOT = candidate
        break


# ----------------------------------------------------------
# Only mount if Drive is not already available.
# Uses a separate safe mountpoint.
# ----------------------------------------------------------

if DRIVE_ROOT is None:

    from google.colab import drive

    SAFE_MOUNTPOINT = "/content/gdrive_safe"

    # Refuse to touch a non-empty directory.
    if os.path.exists(SAFE_MOUNTPOINT):
        contents = os.listdir(SAFE_MOUNTPOINT)

        if len(contents) > 0:
            raise RuntimeError(
                f"STOP: {SAFE_MOUNTPOINT} already contains files.\n"
                "Nothing has been deleted or modified."
            )

    drive.mount(SAFE_MOUNTPOINT)

    DRIVE_ROOT = os.path.join(
        SAFE_MOUNTPOINT,
        "MyDrive"
    )


print("Google Drive root:")
print(DRIVE_ROOT)

In [ ]:
# ==========================================================
# CELL 3 — LOCK PROJECT PATH
# ==========================================================

PROJECT_DIR = os.path.join(
    DRIVE_ROOT,
    "RAG_ICH_MASTER"
)

PROJECT_DIR = os.path.realpath(
    PROJECT_DIR
)

print("Expected project folder:")
print(PROJECT_DIR)


# ----------------------------------------------------------
# IMPORTANT:
# Do NOT create the project folder automatically.
# If it is missing, STOP.
# ----------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):

    raise FileNotFoundError(
        "\nSTOP.\n\n"
        "RAG_ICH_MASTER was not found at:\n"
        f"{PROJECT_DIR}\n\n"
        "No folders have been created and "
        "nothing has been deleted.\n"
        "Check Google Drive before continuing."
    )


if os.path.basename(PROJECT_DIR) != "RAG_ICH_MASTER":

    raise RuntimeError(
        "Safety check failed: unexpected project path."
    )


print("\n✓ RAG_ICH_MASTER confirmed.")
print("✓ Safe to continue.")

In [ ]:
# ==========================================================
# CELL 4 — INSPECT PROJECT
# ==========================================================

print("Contents of RAG_ICH_MASTER:\n")

for item in sorted(os.listdir(PROJECT_DIR)):
    print(item)

In [ ]:
# ==========================================================
# CELL 5 — IMPORTS
# ==========================================================

import os
import glob
import gc
import json
import time
import random

from pathlib import Path
from datetime import datetime

import pandas as pd
import torch

from groq import Groq

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Imports successful.")

In [ ]:
# ==========================================================
# CELL 6 — FIND RECURSIVE CHUNKS SAFELY
# ==========================================================

chunk_matches = glob.glob(
    os.path.join(
        PROJECT_DIR,
        "**",
        "recursive_chunks.csv"
    ),
    recursive=True
)

print(
    f"Found {len(chunk_matches)} "
    "recursive_chunks.csv file(s):\n"
)

for i, path in enumerate(chunk_matches):
    print(f"[{i}] {path}")


if len(chunk_matches) == 0:

    raise FileNotFoundError(
        "\nSTOP: recursive_chunks.csv was not found "
        "inside RAG_ICH_MASTER."
    )


elif len(chunk_matches) > 1:

    raise RuntimeError(
        "\nSTOP: More than one recursive_chunks.csv "
        "was found.\n"
        "Nothing has been changed.\n\n"
        "Review the paths printed above and manually "
        "choose the correct one."
    )


CHUNK_PATH = chunk_matches[0]

print("\nSelected:")
print(CHUNK_PATH)

In [ ]:
# ==========================================================
# CELL 7 — VALIDATE RECURSIVE CHUNKS
# ==========================================================

chunk_df = pd.read_csv(
    CHUNK_PATH
)

required_chunk_columns = {
    "chunk_id",
    "text"
}

missing_columns = (
    required_chunk_columns
    - set(chunk_df.columns)
)

if missing_columns:

    raise ValueError(
        "Chunk file is missing required columns: "
        f"{missing_columns}"
    )


print("Chunk file loaded successfully.")
print("Path:", CHUNK_PATH)
print("Number of chunks:", len(chunk_df))
print("Columns:", chunk_df.columns.tolist())

display(
    chunk_df.head()
)

In [ ]:
# ==========================================================
# CELL 8 — FIND EVALUATION DATASET SAFELY
# ==========================================================

eval_matches = glob.glob(
    os.path.join(
        PROJECT_DIR,
        "**",
        "evaluation_questions.xlsx"
    ),
    recursive=True
)

print(
    f"Found {len(eval_matches)} "
    "evaluation_questions.xlsx file(s):\n"
)

for i, path in enumerate(eval_matches):
    print(f"[{i}] {path}")


if len(eval_matches) == 0:

    raise FileNotFoundError(
        "\nSTOP: evaluation_questions.xlsx "
        "was not found inside RAG_ICH_MASTER."
    )


elif len(eval_matches) > 1:

    raise RuntimeError(
        "\nSTOP: More than one evaluation_questions.xlsx "
        "was found.\n"
        "Nothing has been modified.\n"
        "Review the paths printed above."
    )


EVAL_FILE = eval_matches[0]

print("\nSelected:")
print(EVAL_FILE)

In [ ]:
# ==========================================================
# CELL 9 — LOAD + VALIDATE EVALUATION DATASET
# ==========================================================

evaluation_df = pd.read_excel(
    EVAL_FILE
)

required_eval_columns = [
    "question_id",
    "domain",
    "subdomain",
    "clinical_question",
    "reference_answer_draft",
    "gold_evidence_location",
    "required_gold_evidence_for_retrieval"
]

missing = [
    column
    for column in required_eval_columns
    if column not in evaluation_df.columns
]

if missing:
    raise ValueError(
        "Evaluation file is missing columns:\n"
        + "\n".join(missing)
    )

if len(evaluation_df) != 29:
    raise ValueError(
        f"STOP: Expected 29 evaluation questions, found {len(evaluation_df)}."
    )

if evaluation_df["question_id"].astype(str).duplicated().any():
    raise ValueError(
        "STOP: Duplicate question_id values were found in the evaluation dataset."
    )

print("Evaluation dataset loaded successfully.")
print("Questions:", len(evaluation_df))
print("Columns:")
print(evaluation_df.columns.tolist())
print("✓ 29 unique evaluation questions confirmed.")

display(
    evaluation_df.head()
)

In [ ]:
from langchain_community.retrievers import BM25Retriever

In [ ]:
# ==========================================================
# FINAL OPTIMISED CONFIGURATION
# ==========================================================

CHUNKING_METHOD = "recursive"

EMBEDDING_MODEL_NAME = (
    "FremyCompany/BioLORD-2023-C"
)

RETRIEVAL_METHOD = "hybrid"

TOP_K = 5

RRF_CONSTANT = 60

GENERATOR_MODEL = "openai/gpt-oss-120b"

TEMPERATURE = 0.0

MAX_COMPLETION_TOKENS = 500


print("FINAL OPTIMISED RAG CONFIGURATION")
print("---------------------------------")
print("Chunking       :", CHUNKING_METHOD)
print("Embedding      :", EMBEDDING_MODEL_NAME)
print("Retrieval      :", RETRIEVAL_METHOD)
print("Top-k          :", TOP_K)
print("RRF constant   :", RRF_CONSTANT)
print("Generator      :", GENERATOR_MODEL)

In [ ]:
# ==========================================================
# BUILD LANGCHAIN DOCUMENTS FROM RECURSIVE CHUNKS
# ==========================================================

documents = []

for _, row in chunk_df.iterrows():
    documents.append(
        Document(
            page_content=str(row["text"]),
            metadata={
                "chunk_id": str(row["chunk_id"]),
                "chunking_method": CHUNKING_METHOD,
                "embedding_model": EMBEDDING_MODEL_NAME
            }
        )
    )

if len(documents) != len(chunk_df):
    raise RuntimeError("STOP: Document count does not match chunk count.")

print("Documents created:", len(documents))


In [ ]:
# ==========================================================
# LOAD BIOLORD EMBEDDING MODEL SAFELY
# ==========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={
        "device": DEVICE
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding device:", DEVICE)
print("Embedding model:", EMBEDDING_MODEL_NAME)


In [ ]:
# ==========================================================
# LOAD EXISTING BIOLORD VECTOR DATABASE
# READ/USE ONLY — NO DELETION
# ==========================================================

STAGE4_VECTOR_DIR = os.path.join(
    PROJECT_DIR,
    "vector_db",
    "stage4_safe"
)

COLLECTION_NAME = (
    "stage4_recursive_biolord_dense_v1"
)

vector_db = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=STAGE4_VECTOR_DIR,
    embedding_function=embedding_model
)

vector_count = (
    vector_db
    ._collection
    .count()
)

print("Vector count:", vector_count)
print("Expected:", len(documents))


if vector_count != len(documents):

    raise RuntimeError(
        "\nSTOP.\n"
        f"Chroma contains {vector_count} vectors "
        f"but {len(documents)} chunks were expected.\n"
        "Nothing has been deleted or changed."
    )


print("\n✓ Existing BioLORD vector database validated.")# ----------------------------------------------------------
# Verify that the stored Chroma metadata corresponds to the
# same recursive chunk file, not merely the same row count.
# ----------------------------------------------------------

stored = vector_db._collection.get(
    include=["metadatas"]
)

stored_chunk_ids = {
    str(meta.get("chunk_id"))
    for meta in stored.get("metadatas", [])
    if meta is not None
}

expected_chunk_ids = {
    str(doc.metadata["chunk_id"])
    for doc in documents
}

if stored_chunk_ids != expected_chunk_ids:
    raise RuntimeError(
        "\nSTOP.\n"
        "The Chroma collection count matches, but its chunk IDs do not "
        "match recursive_chunks.csv.\n"
        "Nothing has been deleted or overwritten."
    )

print("✓ Chroma chunk IDs match recursive_chunks.csv.")


In [ ]:
# ==========================================================
# BM25 RETRIEVER
# ==========================================================

bm25_retriever = BM25Retriever.from_documents(
    documents
)

bm25_retriever.k = TOP_K

print(
    "BM25 retriever ready.",
    "Candidate depth:",
    bm25_retriever.k
)

In [ ]:
# ==========================================================
# DENSE BIOLORD RETRIEVER
# ==========================================================

dense_retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": TOP_K
    }
)

print(
    "Dense BioLORD retriever ready.",
    "Candidate depth:",
    TOP_K
)

In [ ]:
# ==========================================================
# RECIPROCAL RANK FUSION
# ==========================================================

def reciprocal_rank_fusion(
    bm25_docs,
    dense_docs,
    rrf_k=60,
    top_k=5
):

    rrf_scores = {}

    document_lookup = {}


    # ------------------------------
    # BM25 rankings
    # ------------------------------

    for rank, doc in enumerate(
        bm25_docs,
        start=1
    ):

        chunk_id = str(
            doc.metadata["chunk_id"]
        )

        document_lookup[
            chunk_id
        ] = doc

        rrf_scores[
            chunk_id
        ] = (
            rrf_scores.get(
                chunk_id,
                0
            )
            +
            1 / (
                rrf_k
                + rank
            )
        )


    # ------------------------------
    # Dense rankings
    # ------------------------------

    for rank, doc in enumerate(
        dense_docs,
        start=1
    ):

        chunk_id = str(
            doc.metadata["chunk_id"]
        )

        document_lookup[
            chunk_id
        ] = doc

        rrf_scores[
            chunk_id
        ] = (
            rrf_scores.get(
                chunk_id,
                0
            )
            +
            1 / (
                rrf_k
                + rank
            )
        )


    ranked_chunk_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )


    final_docs = [

        document_lookup[
            chunk_id
        ]

        for chunk_id
        in ranked_chunk_ids[
            :top_k
        ]

    ]


    return final_docs

In [ ]:
# ==========================================================
# FINAL HYBRID RETRIEVAL FUNCTION
# ==========================================================

def hybrid_retrieve(
    question,
    top_k=TOP_K,
    rrf_k=RRF_CONSTANT
):

    bm25_docs = (
        bm25_retriever
        .invoke(question)
    )

    dense_docs = (
        dense_retriever
        .invoke(question)
    )

    final_docs = (
        reciprocal_rank_fusion(
            bm25_docs=bm25_docs,
            dense_docs=dense_docs,
            rrf_k=rrf_k,
            top_k=top_k
        )
    )

    return final_docs

In [ ]:
# ==========================================================
# HYBRID RETRIEVAL SANITY TEST
# ==========================================================

test_question = (
    evaluation_df.iloc[0][
        "clinical_question"
    ]
)

hybrid_docs = hybrid_retrieve(
    test_question
)

print("QUESTION:")
print(test_question)

print(
    "\nFinal hybrid chunks:",
    len(hybrid_docs)
)

for rank, doc in enumerate(
    hybrid_docs,
    start=1
):
    print("\n" + "=" * 70)

    print(
        f"Hybrid rank {rank}"
    )

    print(
        "Chunk ID:",
        doc.metadata.get(
            "chunk_id"
        )
    )

    print(
        doc.page_content[:500]
    )


# ==========================================================
# VALIDATION
# ==========================================================

if len(hybrid_docs) != TOP_K:
    raise RuntimeError(
        f"STOP: Hybrid retrieval returned "
        f"{len(hybrid_docs)} chunks; "
        f"expected {TOP_K}."
    )


retrieved_ids = [
    str(
        doc.metadata.get(
            "chunk_id"
        )
    )
    for doc in hybrid_docs
]


if len(set(retrieved_ids)) != TOP_K:
    raise RuntimeError(
        "STOP: Duplicate chunk IDs were found "
        "in the final hybrid top-k."
    )


print(
    "\n✓ Hybrid retrieval returned "
    "5 unique chunks."
)

In [ ]:
# ==========================================================
# GROQ API CLIENT + MODEL ACCESS CHECK
# ==========================================================

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError("STOP: GROQ_API_KEY was not loaded from Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

available_models = {
    model.id
    for model in client.models.list().data
}

if GENERATOR_MODEL not in available_models:
    raise RuntimeError(
        f"STOP: {GENERATOR_MODEL} is not available to this Groq API key."
    )

print("✓ Groq client ready.")
print("✓ Model access confirmed:", GENERATOR_MODEL)


In [ ]:
# ==========================================================
# FIXED STAGE 4 PROMPTS
# ==========================================================

RAG_SYSTEM_PROMPT = """
You are a clinical assistant answering questions about spontaneous intracerebral haemorrhage.
Use only the provided guideline excerpts.
Do not add information from outside the excerpts.
If the answer is not supported by the excerpts, state that the guideline excerpts provided are insufficient.
""".strip()

BASELINE_SYSTEM_PROMPT = """
You are a clinical assistant answering questions about spontaneous intracerebral haemorrhage.
Provide a concise clinical answer.
If uncertain, state your uncertainty.
""".strip()

print("✓ RAG and baseline system prompts loaded.")


In [ ]:
# ==========================================================
# FORMAT RETRIEVED CONTEXT
# ==========================================================

def format_context(docs):
    formatted = []

    for rank, doc in enumerate(docs, start=1):
        chunk_id = doc.metadata.get("chunk_id", "unknown")
        formatted.append(
            f"[Excerpt {rank} | Chunk {chunk_id}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted)


In [ ]:
# ==========================================================
# BUILD USER PROMPTS
# ==========================================================

def build_rag_user_prompt(question, context):
    return f"""
Clinical question:

{question}

Guideline excerpts:

{context}

Task:

Provide a concise clinical answer based only on the guideline excerpts above.

Include recommendation strength or level of evidence where it is provided in the excerpts.

Do not fabricate recommendations, drug doses, thresholds, or other clinical details that are not present in the excerpts.
""".strip()


def build_baseline_user_prompt(question):
    return f"""
Clinical question:

{question}

Task:

Provide a concise clinical answer.
""".strip()


In [ ]:
# ==========================================================
# SAFE GROQ GENERATION FUNCTION
# ==========================================================

def call_groq(system_prompt, user_prompt, max_retries=6):

    for attempt in range(max_retries):
        try:
            start_time = time.perf_counter()

            completion = client.chat.completions.create(
                model=GENERATOR_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                temperature=TEMPERATURE,
                max_completion_tokens=MAX_COMPLETION_TOKENS
            )

            latency = time.perf_counter() - start_time

            answer = completion.choices[0].message.content
            if answer is None:
                answer = ""

            return answer.strip(), latency

        except Exception as error:
            status_code = getattr(error, "status_code", None)

            # Permanent request/auth/model errors should not be retried.
            if status_code in {400, 401, 403, 404}:
                print("\nPermanent Groq API error:")
                print(error)
                raise

            if attempt == max_retries - 1:
                print("\nMaximum retries reached.")
                raise

            wait_seconds = min(2 ** attempt, 30)

            print(
                f"\nTemporary API error on attempt {attempt + 1}: {error}"
            )
            print(f"Retrying in {wait_seconds} seconds...")

            time.sleep(wait_seconds)


In [ ]:
# ==========================================================
# TEST ONE HYBRID RAG ANSWER
# ==========================================================

test_question = (
    evaluation_df.iloc[0][
        "clinical_question"
    ]
)

test_docs = hybrid_retrieve(
    test_question
)

test_context = format_context(
    test_docs
)

test_prompt = build_rag_user_prompt(
    test_question,
    test_context
)

test_answer, test_latency = call_groq(
    RAG_SYSTEM_PROMPT,
    test_prompt
)


print("=" * 70)
print("QUESTION")
print("=" * 70)
print(test_question)

print("\n" + "=" * 70)
print("HYBRID RAG ANSWER")
print("=" * 70)
print(test_answer)

print(
    f"\nLatency: "
    f"{test_latency:.2f} seconds"
)

In [ ]:
# ==========================================================
# FINAL HYBRID STAGE 4 OUTPUT FOLDER
# ==========================================================

FINAL_HYBRID_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "stage4_FINAL_HYBRID"
)

os.makedirs(
    FINAL_HYBRID_DIR,
    exist_ok=True
)

print(
    "Final Hybrid Stage 4 outputs:"
)

print(
    FINAL_HYBRID_DIR
)

In [ ]:
# ==========================================================
# FINAL HYBRID-RAG GENERATION
# ==========================================================

HYBRID_RAG_OUTPUT = os.path.join(
    FINAL_HYBRID_DIR,
    "hybrid_rag_answers_gpt_oss_120b.csv"
)


if os.path.exists(
    HYBRID_RAG_OUTPUT
):

    hybrid_results_df = pd.read_csv(
        HYBRID_RAG_OUTPUT
    )

    hybrid_results = (
        hybrid_results_df
        .to_dict("records")
    )

    completed_questions = set(
        hybrid_results_df[
            "question_id"
        ].astype(str)
    )

    print(
        f"Resuming: "
        f"{len(completed_questions)} "
        "questions already completed."
    )


else:

    hybrid_results = []

    completed_questions = set()

    print(
        "Starting new FINAL Hybrid-RAG generation."
    )


for _, row in evaluation_df.iterrows():

    question_id = str(
        row["question_id"]
    )

    if question_id in completed_questions:

        print(
            f"{question_id}: skipped"
        )

        continue


    question = row[
        "clinical_question"
    ]


    print(
        f"\nGenerating FINAL Hybrid-RAG: "
        f"{question_id}"
    )


    # --------------------------------
    # HYBRID RETRIEVAL
    # --------------------------------

    docs = hybrid_retrieve(
        question
    )


    context = format_context(
        docs
    )


    chunk_ids = [

        str(
            doc.metadata.get(
                "chunk_id"
            )
        )

        for doc in docs
    ]


    # --------------------------------
    # GENERATION
    # --------------------------------

    user_prompt = (
        build_rag_user_prompt(
            question,
            context
        )
    )


    answer, latency = call_groq(
        RAG_SYSTEM_PROMPT,
        user_prompt
    )


    result = {

        "question_id":
            question_id,

        "domain":
            row["domain"],

        "subdomain":
            row["subdomain"],

        "clinical_question":
            question,

        "reference_answer_draft":
            row[
                "reference_answer_draft"
            ],

        "gold_evidence_location":
            row[
                "gold_evidence_location"
            ],

        "required_gold_evidence_for_retrieval":
            row[
                "required_gold_evidence_for_retrieval"
            ],

        "retrieved_chunk_ids":
            "|".join(
                chunk_ids
            ),

        "retrieved_context":
            context,

        "generated_answer":
            answer,

        "latency_seconds":
            round(
                latency,
                3
            ),

        "condition":
            "RAG",

        "chunking_method":
            CHUNKING_METHOD,

        "embedding_model":
            EMBEDDING_MODEL_NAME,

        "retrieval_method":
            RETRIEVAL_METHOD,

        "rrf_constant":
            RRF_CONSTANT,

        "top_k":
            TOP_K,

        "generator_model":
            GENERATOR_MODEL
    }


    hybrid_results.append(
        result
    )


    pd.DataFrame(
        hybrid_results
    ).to_csv(
        HYBRID_RAG_OUTPUT,
        index=False
    )


    completed_questions.add(
        question_id
    )


    print(
        f"{question_id}: saved"
    )


    time.sleep(5)


hybrid_results_df = pd.DataFrame(
    hybrid_results
)


print(
    "\nFINAL Hybrid-RAG responses:",
    len(hybrid_results_df)
)

In [ ]:
# ==========================================================
# FRESH FINAL BASELINE GENERATION
# ==========================================================

FINAL_BASELINE_OUTPUT = os.path.join(
    FINAL_HYBRID_DIR,
    "baseline_answers_gpt_oss_120b.csv"
)


if os.path.exists(
    FINAL_BASELINE_OUTPUT
):

    baseline_results_df = pd.read_csv(
        FINAL_BASELINE_OUTPUT
    )

    baseline_results = (
        baseline_results_df
        .to_dict("records")
    )

    completed_questions = set(
        baseline_results_df[
            "question_id"
        ].astype(str)
    )


else:

    baseline_results = []

    completed_questions = set()


for _, row in evaluation_df.iterrows():

    question_id = str(
        row["question_id"]
    )

    if question_id in completed_questions:

        print(
            f"{question_id}: skipped"
        )

        continue


    question = row[
        "clinical_question"
    ]


    print(
        f"\nGenerating FINAL baseline: "
        f"{question_id}"
    )


    user_prompt = (
        build_baseline_user_prompt(
            question
        )
    )


    answer, latency = call_groq(
        BASELINE_SYSTEM_PROMPT,
        user_prompt
    )


    result = {

        "question_id":
            question_id,

        "domain":
            row["domain"],

        "subdomain":
            row["subdomain"],

        "clinical_question":
            question,

        "reference_answer_draft":
            row[
                "reference_answer_draft"
            ],

        "gold_evidence_location":
            row[
                "gold_evidence_location"
            ],

        "required_gold_evidence_for_retrieval":
            row[
                "required_gold_evidence_for_retrieval"
            ],

        "generated_answer":
            answer,

        "latency_seconds":
            round(
                latency,
                3
            ),

        "condition":
            "Baseline",

        "generator_model":
            GENERATOR_MODEL
    }


    baseline_results.append(
        result
    )


    pd.DataFrame(
        baseline_results
    ).to_csv(
        FINAL_BASELINE_OUTPUT,
        index=False
    )


    completed_questions.add(
        question_id
    )


    print(
        f"{question_id}: saved"
    )


    time.sleep(5)


baseline_results_df = pd.DataFrame(
    baseline_results
)


print(
    "\nFINAL baseline responses:",
    len(baseline_results_df)
)

In [ ]:
assert len(hybrid_results_df) == 29
assert len(baseline_results_df) == 29

print("✓ Hybrid RAG = 29")
print("✓ Baseline   = 29")
print("✓ Total      = 58")

In [ ]:
combined_final_df = pd.concat(
    [
        hybrid_results_df,
        baseline_results_df
    ],
    ignore_index=True,
    sort=False
)

FINAL_RAW_FILE = os.path.join(
    FINAL_HYBRID_DIR,
    "FINAL_hybrid_rag_vs_baseline_raw.csv"
)

combined_final_df.to_csv(
    FINAL_RAW_FILE,
    index=False
)

print(
    "Combined responses:",
    len(combined_final_df)
)

In [ ]:
# ==========================================================
# NEW FINAL BLINDING
# ==========================================================

final_blinded_df = (
    combined_final_df
    .sample(
        frac=1,
        random_state=20260825
    )
    .reset_index(
        drop=True
    )
)


final_blinded_df.insert(
    0,
    "response_id",
    [
        f"S{i:03d}"
        for i in range(
            1,
            len(final_blinded_df) + 1
        )
    ]
)

print(
    "Final blinded responses:",
    len(final_blinded_df)
)

In [ ]:
FINAL_KEY_FILE = os.path.join(
    FINAL_HYBRID_DIR,
    "FINAL_unblinding_key.csv"
)


final_key_df = final_blinded_df[
    [
        "response_id",
        "question_id",
        "condition"
    ]
].copy()


final_key_df.to_csv(
    FINAL_KEY_FILE,
    index=False
)


print(
    "Unblinding key saved separately:"
)

print(
    FINAL_KEY_FILE
)

In [ ]:
# ==========================================================
# FINAL BLINDED SCORING WORKBOOK
# ==========================================================

final_scoring_df = final_blinded_df[
    [
        "response_id",
        "question_id",
        "domain",
        "subdomain",
        "clinical_question",
        "reference_answer_draft",
        "gold_evidence_location",
        "required_gold_evidence_for_retrieval",
        "generated_answer"
    ]
].copy()


final_scoring_df[
    "faithfulness_0_2"
] = ""

final_scoring_df[
    "clinical_correctness_0_2"
] = ""

final_scoring_df[
    "completeness_0_2"
] = ""

final_scoring_df[
    "conciseness_0_1"
] = ""

final_scoring_df[
    "unsupported_claims_yes_no"
] = ""

final_scoring_df[
    "evaluator_comments"
] = ""


FINAL_BLINDED_FILE = os.path.join(
    FINAL_HYBRID_DIR,
    "FINAL_HYBRID_blinded_evaluation.xlsx"
)


if os.path.exists(FINAL_BLINDED_FILE):
    raise FileExistsError(
        "\nSTOP: The blinded evaluation workbook already exists:\n"
        f"{FINAL_BLINDED_FILE}\n\n"
        "It will NOT be overwritten. This protects any manual scores "
        "that may already have been entered."
    )

final_scoring_df.to_excel(
    FINAL_BLINDED_FILE,
    index=False
)

print(
    "Final blinded workbook:"
)

print(
    FINAL_BLINDED_FILE
)


In [ ]:
# ==========================================================
# SAVE FINAL HYBRID EXPERIMENT LOG
# ==========================================================

EXPERIMENT_LOG_FILE = os.path.join(
    FINAL_HYBRID_DIR,
    "FINAL_HYBRID_experiment_log.json"
)

experiment_log = {
    "experiment": "Final Hybrid RAG vs Non-RAG Evaluation",
    "project_folder": "RAG_ICH_MASTER",
    "chunking_strategy": CHUNKING_METHOD,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "retrieval_method": RETRIEVAL_METHOD,
    "top_k": TOP_K,
    "rrf_constant": RRF_CONSTANT,
    "generator_originally_planned": "llama-3.3-70b-versatile",
    "generator_implemented": GENERATOR_MODEL,
    "generator_change_reason": (
        "The originally planned llama-3.3-70b-versatile model was not "
        "available to the active Groq API project at the time of the final experiment."
    ),
    "temperature": TEMPERATURE,
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "number_of_questions": len(evaluation_df),
    "rag_system_prompt": RAG_SYSTEM_PROMPT,
    "baseline_system_prompt": BASELINE_SYSTEM_PROMPT,
    "stage1_selection": {
        "selected": "recursive",
        "recall_at_5": 0.503448276,
        "mrr_at_5": 0.713
    },
    "stage2_selection": {
        "selected": "FremyCompany/BioLORD-2023-C",
        "recall_at_5": 0.55862069,
        "mrr_at_5": 0.721
    },
    "stage3_selection": {
        "selected": "hybrid",
        "hybrid_recall_at_5": 0.606896552,
        "hybrid_mrr_at_5": 0.868,
        "dense_recall_at_5": 0.620689655,
        "dense_mrr_at_5": 0.830
    },
    "timestamp": datetime.now().astimezone().isoformat()
}

with open(
    EXPERIMENT_LOG_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_log,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Experiment log saved:")
print(EXPERIMENT_LOG_FILE)


In [ ]:
assert (
    "condition"
    not in final_scoring_df.columns
)

assert (
    "retrieval_method"
    not in final_scoring_df.columns
)

assert (
    "retrieved_context"
    not in final_scoring_df.columns
)

print(
    "✓ Condition hidden"
)

print(
    "✓ Retrieval method hidden"
)

print(
    "✓ Retrieved context hidden"
)

print(
    "✓ Final scoring workbook ready"
)

In [ ]:
import os
import pandas as pd

FINAL_HYBRID_DIR = (
    "/content/drive/MyDrive/RAG_ICH_MASTER/"
    "results/stage4_FINAL_HYBRID"
)

SCORED_FILE = os.path.join(
    FINAL_HYBRID_DIR,
    "FINAL_HYBRID_blinded_evaluation.xlsx"
)

scored_df = pd.read_excel(SCORED_FILE)

print("Rows:", len(scored_df))
print(scored_df.columns.tolist())